# TurboLoader quickstart — fast image loading for PyTorch

[TurboLoader](https://github.com/ALJainProjects/TurboLoader) is a C++20 data loader:
one parallel pass decodes → resizes → augments → normalizes straight into the output batch,
with SIMD everywhere and the GIL released. This notebook, on **real ImageNet images
(Imagenette)**, shows:

1. loading speed vs a PIL loop,
2. the fused train pipeline (`train_aug=True`: RandomResizedCrop + flip + normalize),
3. feeding PyTorch with pinned recycled buffers + mid-epoch checkpointing.

Works on Kaggle/Colab CPU kernels (Linux x86_64 wheels) — no GPU required.

In [1]:
# On Kaggle/Colab:
# %pip install -q turboloader
import turboloader as tl
import numpy as np
print("turboloader", tl.__version__)

turboloader 2.32.1.dev4+g9a29ba224.d20260709


In [2]:
# Real data: Imagenette-160 (9,469 real ImageNet JPEGs), packed into a TAR.
import os, glob, tarfile, urllib.request

DATA = os.environ.get("IMAGENETTE_DIR", "/tmp/imagenette2-160")
TAR = "/tmp/imagenette_train.tar"
if not os.path.isdir(DATA):
    tgz = "/tmp/imagenette2-160.tgz"
    urllib.request.urlretrieve(
        "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", tgz)
    import tarfile as _tf
    with _tf.open(tgz) as f:
        f.extractall("/tmp")
if not os.path.exists(TAR):
    paths = sorted(glob.glob(os.path.join(DATA, "train/*/*.JPEG")))
    with tarfile.open(TAR, "w") as tf:
        for i, p in enumerate(paths):
            tf.add(p, arcname=f"{i:06d}.jpg")
print("images:", len(glob.glob(os.path.join(DATA, 'train/*/*.JPEG'))))

images: 9469


In [3]:
# 1) Loading speed: TurboLoader fused pass vs a PIL loop, same work
import time
from PIL import Image

loader = tl.DataLoader(TAR, batch_size=64, output_format="pytorch", image_size=160,
                       transform=tl.Resize(160,160) | tl.ImageNetNormalize(), shuffle=False)
t0 = time.perf_counter(); n = 0
for images, meta in loader:
    n += images.shape[0]
tl_rate = n / (time.perf_counter() - t0)

MEAN = np.array([0.485,0.456,0.406], np.float32); STD = np.array([0.229,0.224,0.225], np.float32)
paths = sorted(glob.glob(os.path.join(DATA, "train/*/*.JPEG")))[:512]
t0 = time.perf_counter()
for p in paths:
    a = np.asarray(Image.open(p).convert("RGB").resize((160,160)), np.float32) / 255.0
    a = (a - MEAN) / STD
pil_rate = len(paths) / (time.perf_counter() - t0)
print(f"TurboLoader: {tl_rate:,.0f} img/s   |   PIL loop: {pil_rate:,.0f} img/s   "
      f"|   speedup {tl_rate/pil_rate:.1f}x")

TurboLoader: 58,699 img/s   |   PIL loop: 2,474 img/s   |   speedup 23.7x


In [4]:
# 2) Fused train augmentation: RandomResizedCrop + flip + normalize in the same C++ pass
aug = tl.DataLoader(TAR, batch_size=64, output_format="pytorch", image_size=160,
                    transform=tl.ImageNetNormalize(), shuffle=True, seed=0,
                    train_aug=True, prefetch_batches=4)
t0 = time.perf_counter(); n = 0
for images, meta in aug:
    n += images.shape[0]
print(f"train-augmented epoch: {n} imgs at {n/(time.perf_counter()-t0):,.0f} img/s")
print("deterministic per (seed, epoch): aug.set_epoch(k) reproduces exactly")

train-augmented epoch: 9469 imgs at 55,369 img/s
deterministic per (seed, epoch): aug.set_epoch(k) reproduces exactly


In [5]:
# 3) PyTorch integration: pinned recycled buffers + exact mid-epoch resume
try:
    import torch
    ld = tl.DataLoader(TAR, batch_size=64, output_format="pytorch", image_size=160,
                       transform=tl.ImageNetNormalize(), shuffle=True, seed=0,
                       train_aug=True, pin_memory=True, prefetch_batches=4)
    it = iter(ld)
    x, meta = next(it)
    print("batch:", type(x).__name__, tuple(x.shape), x.dtype,
          "| pinned:", x.is_pinned() if torch.cuda.is_available() else "(needs CUDA)")
    sd = ld.state_dict()          # {'epoch': 0, 'batches_served': 1}
    print("checkpoint:", sd, "-> load_state_dict(sd) resumes exactly here, decode-free")
except ImportError:
    print("pip install torch for this section")

batch: Tensor (64, 3, 160, 160) torch.float32 | pinned: (needs CUDA)
checkpoint: {'version': 1, 'epoch': 0, 'batches_served': 1} -> load_state_dict(sd) resumes exactly here, decode-free
